# Notebook 07 — A Decoder Transformer from Scratch

    ## Learning objectives

    - Track tensor shapes through embeddings, attention, MLP, residuals, and LM head
- Implement causal self-attention and a pre-norm decoder block
- Calculate the dominant parameter and compute terms

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 7.1 Shape ledger

Let batch \(B\), sequence \(T\), hidden width \(D\), heads \(H\), and head width
\(d_h=D/H\). Token embeddings have shape `[B,T,D]`. Projections produce Q, K, V;
reshaping gives `[B,H,T,d_h]`. Attention scores are `[B,H,T,T]`. The MLP usually
expands width by roughly 3–4×. Residual paths require matching `[B,T,D]` shapes.


In [ ]:
import math
import torch
from torch import nn

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dh = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def heads(t): return t.view(B, T, self.h, self.dh).transpose(1, 2)
        q, k, v = map(heads, (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dh)
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        weights = scores.masked_fill(mask, float("-inf")).softmax(-1)
        y = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.out(y), weights


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    def forward(self, x):
        scale = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * scale).to(x.dtype) * self.weight

class DecoderBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, expansion=4):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads)
        self.norm2 = RMSNorm(d_model)
        hidden = expansion * d_model
        self.mlp = nn.Sequential(nn.Linear(d_model, hidden), nn.GELU(),
                                 nn.Linear(hidden, d_model))

    def forward(self, x):
        a, weights = self.attn(self.norm1(x))
        x = x + a
        x = x + self.mlp(self.norm2(x))
        return x, weights

block = DecoderBlock()
x = torch.randn(2, 8, 64)
y, attention = block(x)
print("output", y.shape, "attention", attention.shape)
print("parameters", sum(p.numel() for p in block.parameters()))


## 7.2 Why scaling, residuals, and normalization matter

Dot products grow in magnitude with head dimension; division by \(\sqrt{d_h}\) keeps
softmax from saturating early. Residual connections create short gradient paths.
Pre-norm architectures normalize before each sublayer and are generally easier to train
deeply. RMSNorm rescales by root-mean-square without subtracting the mean.

Dense self-attention materializes \(T^2\) scores. The MLP often dominates parameters
and FLOPs at shorter contexts; attention becomes dominant as context grows.


## 7.3 Complete decoder data flow

A decoder-only transformer performs: token lookup → positional transformation → repeated
decoder blocks → final normalization → vocabulary projection. With tied embeddings, the
output matrix is the transpose of the input embedding matrix, reducing parameters and
encouraging a shared lexical geometry. Each block contains a communication operation
(causal attention) and a per-position computation operation (MLP).

Modern MLPs often use gated activations such as SwiGLU rather than a two-layer GELU MLP:
`down(silu(gate(x)) * up(x))`. Gating increases projection parameters but often improves
quality. Biases may be omitted. RMSNorm is common. Dropout is frequently zero in large-model
pretraining. Architectural labels such as “Llama-like” still hide important choices: head
counts, GQA groups, RoPE base/scaling, vocabulary, tie policy, activation, norm epsilon,
initialization, and local/sliding attention.


In [ ]:
# Assemble a tiny causal LM from the previously defined block.
class TinyDecoderLM(nn.Module):
    def __init__(self, vocab=128, d_model=64, layers=3, heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, heads) for _ in range(layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.embed.weight  # tied weights
    def forward(self, ids):
        x = self.embed(ids)
        maps = []
        for block in self.blocks:
            x, attention = block(x); maps.append(attention)
        return self.lm_head(self.final_norm(x)), maps

tiny = TinyDecoderLM()
ids = torch.randint(0, 128, (2, 12))
logits, maps = tiny(ids)
labels = ids.clone()
loss = nn.functional.cross_entropy(logits[:, :-1].reshape(-1, 128), labels[:, 1:].reshape(-1))
loss.backward()
print("logits", logits.shape, "loss", loss.item())
print("tied storage:", tiny.lm_head.weight.data_ptr() == tiny.embed.weight.data_ptr())


## 7.4 Parameter and compute accounting

Ignoring biases, standard attention projections contain roughly \(4D^2\) parameters
(Q, K, V, output). A conventional 4× MLP contains roughly \(8D^2\). Thus one block is
about \(12D^2\), before embeddings and norms. GQA reduces K/V projection parameters but
not Q/output. A gated MLP with two input projections and one output projection changes the
count. Embeddings cost \(VD\), which is substantial for large vocabularies and small models.

Training compute is often summarized as approximately six times parameter count times
training tokens for dense transformers, but this is a planning approximation. Actual FLOPs
depend on sequence length, attention, activation checkpointing, sparsity/MoE routing, and
implementation. Inference separates prefill (parallel prompt processing) from decode
(sequential, cache-reading token steps). Parameter count alone does not predict latency.


In [ ]:
def rough_decoder_params(vocab, width, layers, mlp_ratio=4, tied=True):
    embeddings = vocab * width * (1 if tied else 2)
    attention = layers * 4 * width**2
    mlp = layers * 2 * mlp_ratio * width**2
    norms = layers * 2 * width + width
    return {"embeddings": embeddings, "attention": attention, "mlp": mlp,
            "norms": norms, "total": embeddings + attention + mlp + norms}

for config in [(32_000, 768, 12), (128_000, 2048, 24), (128_000, 4096, 32)]:
    parts = rough_decoder_params(*config)
    print(config, {k: f"{v/1e9:.3f}B" for k, v in parts.items()})


## 7.5 Initialization, residual scale, and architecture diagnostics

If activations or residual updates grow with depth, softmax and nonlinearities can saturate
and gradients can destabilize. Initialization scales projection weights; some architectures
scale residual output projections by depth. Pre-norm provides a clean identity path but can
produce large residual streams; post-norm changes optimization behavior. Deep networks also
benefit from careful optimizer warmup and precision choices.

When implementing a block, test invariants before training: shapes for multiple batch/lengths;
no attention above the causal diagonal; no NaNs on extreme but valid inputs; deterministic
forward under eval; gradients reach every intended parameter; padding does not influence
non-padding outputs; cached and uncached generation agree; and a tiny dataset can be
overfit. The “overfit one batch” test is one of the fastest ways to expose target shifting,
masks, detached tensors, or optimizer errors.


## 7.6 Decoder architecture reference

| Component | Input/output | Primary role |
|---|---|---|
| Embedding | IDs `[B,T]` → `[B,T,D]` | Learned token representation |
| Norm | `[B,T,D]` → same | Controls scale/optimization |
| Attention | same → same | Communicates among causally visible positions |
| MLP/SwiGLU | same → same | Per-position nonlinear feature transformation |
| Residual | same + same | Identity/gradient path and feature accumulation |
| LM head | `[B,T,D]` → `[B,T,V]` | Vocabulary logits |

Debug in this order: verify shift/mask; overfit one batch; compare logits for a prefix alone versus
the same prefix followed by hidden future tokens; test padding invariance; inspect activation and
gradient norms by layer; compare train/eval modes; then optimize kernels. A model that cannot overfit
a tiny deterministic sequence has an implementation or optimization problem, not insufficient data.

Architectural configuration is part of checkpoint compatibility. Width, heads/KV heads, head
dimension, layer count, MLP width/activation, norm type/epsilon, RoPE configuration, vocabulary,
tie policy, and attention pattern must match weights. Shape-compatible changes can still silently
destroy behavior.


## 7.7 Parameter accounting and weight tying

A decoder's parameter count is dominated by token embeddings, attention and MLP projections, and sometimes a separate output projection. Compute each term from vocabulary size, width, layer count, and expansion ratio before instantiating the model. Weight tying makes the language-model head reuse the token-embedding matrix, reducing parameters and coupling input and output representations. Verify actual storage identity rather than equal initial values. Bias choices and gated MLPs alter formulas. Parameter count predicts weight memory but not activation, optimizer, KV-cache, temporary-workspace, or fragmentation costs.


In [ ]:
def rough_decoder_params(vocab,width,layers,mlp_ratio=4,tied=True):
 embeddings=vocab*width; attention=layers*4*width*width; mlp=layers*2*width*(mlp_ratio*width); norms=layers*4*width; head=0 if tied else vocab*width
 return {"embeddings":embeddings,"attention":attention,"mlp":mlp,"norms":norms,"head":head,"total":embeddings+attention+mlp+norms+head}
print(rough_decoder_params(32000,768,12))


## 7.8 Residual-stream and initialization checks

Pre-norm decoder blocks normalize before attention and MLP sublayers, then add each result to the residual stream. Post-norm variants place normalization after addition and have different optimization behavior. Trace norms at the embedding output and after every residual update; exploding or collapsing values often reveal initialization, masking, or precision problems. Initialization scale should reflect fan-in and residual depth, and padding embeddings may require special treatment. Before language-model training, assert finite logits, zero probability on causally forbidden positions, correct tied storage, and deterministic evaluation. Then overfit one batch and test checkpoint reload.


In [ ]:
torch.manual_seed(0)
residual=torch.randn(2,5,16); norm=nn.LayerNorm(16); branch=nn.Linear(16,16,bias=False)
with torch.no_grad(): branch.weight.normal_(0,.02)
updated=residual+branch(norm(residual))
print("norms",residual.norm(dim=-1).mean().item(),updated.norm(dim=-1).mean().item()); assert torch.isfinite(updated).all()


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)


## Exercises

    1. Add token embeddings, a final norm, and a tied LM head.
2. Verify all probability mass above the causal diagonal is zero.
3. Estimate parameters for a 24-layer, width-2048 decoder with 4× MLP expansion.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
